In [0]:
%sql
SET spark.databricks.execution.timeout = 14400;

In [0]:
from pyspark.sql.functions import col, upper, coalesce, to_date

# 1. Carregando a tabela que a gente acabou de criar
nome_tabela = "workspace.default.silver_ancine"
df_silver = spark.read.table(nome_tabela)

# 2. As Transformações de Mestre
df_tratado = df_silver \
    .withColumn("DATA_EXIBICAO", to_date(col("DATA_EXIBICAO"), "dd/MM/yyyy")) \
    .withColumn("TITULO_ORIGINAL", upper(col("TITULO_ORIGINAL"))) \
    .withColumn("TITULO_BRASIL", upper(col("TITULO_BRASIL"))) \
    .withColumn("RAZAO_SOCIAL_DISTRIBUIDORA", upper(col("RAZAO_SOCIAL_DISTRIBUIDORA"))) \
    .withColumn("TITULO_BRASIL", coalesce(col("TITULO_BRASIL"), col("TITULO_ORIGINAL")))

# 3. Sobrescrevendo a tabela com os dados lapidados
df_tratado.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(nome_tabela)

# Mostrando como ficou o trabalho final
display(spark.sql(f"SELECT * FROM {nome_tabela} LIMIT 10"))

In [0]:
# Tabela Gold 1: Top Distribuidoras
df_gold_top_distribuidoras = spark.sql("""
    SELECT 
        RAZAO_SOCIAL_DISTRIBUIDORA, 
        SUM(PUBLICO) AS PUBLICO_TOTAL
    FROM workspace.default.silver_ancine
    GROUP BY RAZAO_SOCIAL_DISTRIBUIDORA
    ORDER BY PUBLICO_TOTAL DESC
    LIMIT 3
""")

# Salvando a tabela Gold e mostrando o resultado
df_gold_top_distribuidoras.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_top_distribuidoras")
display(spark.sql("SELECT * FROM workspace.default.gold_top_distribuidoras"))

In [0]:
# Tabela Gold 2: Comparativo de Público por Origem
df_gold_origem = spark.sql("""
    SELECT 
        CASE 
            WHEN PAIS_OBRA = 'BRASIL' THEN 'NACIONAL'
            ELSE 'ESTRANGEIRO' 
        END AS ORIGEM,
        SUM(PUBLICO) AS PUBLICO_TOTAL
    FROM workspace.default.silver_ancine
    GROUP BY 
        CASE 
            WHEN PAIS_OBRA = 'BRASIL' THEN 'NACIONAL'
            ELSE 'ESTRANGEIRO' 
        END
    ORDER BY PUBLICO_TOTAL DESC
""")

df_gold_origem.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_origem_publico")
display(spark.sql("SELECT * FROM workspace.default.gold_origem_publico"))

In [0]:
# Tabela Gold 3: O grande recorde diário
df_gold_recorde = spark.sql("""
    SELECT 
        DATA_EXIBICAO,
        TITULO_BRASIL,
        PAIS_OBRA,
        PUBLICO AS PUBLICO_DIARIO
    FROM workspace.default.silver_ancine
    ORDER BY PUBLICO DESC
    LIMIT 5
""")

df_gold_recorde.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_recorde_diario")
display(spark.sql("SELECT * FROM workspace.default.gold_recorde_diario"))

In [0]:
%sql
-- Joga isso numa célula SQL do Databricks (usa %sql na primeira linha)
CREATE OR REPLACE TABLE workspace.default.gold_ranking_distribuidora AS
SELECT 
    RAZAO_SOCIAL_DISTRIBUIDORA,
    TITULO_BRASIL,
    PUBLICO,
    RANK() OVER(PARTITION BY RAZAO_SOCIAL_DISTRIBUIDORA ORDER BY PUBLICO DESC) as RANKING_INTERNO
FROM workspace.default.silver_ancine
QUALIFY RANKING_INTERNO <= 3;

In [0]:
%sql
-- Mostra o histórico de todas as alterações e versões da tua tabela! A banca pira nisso.
DESCRIBE HISTORY workspace.default.silver_ancine;

In [0]:
# Criando features (variáveis) preparadas para plugar num modelo preditivo
df_features = spark.sql("""
    SELECT 
        TITULO_BRASIL,
        CASE WHEN PAIS_OBRA = 'ESTADOS UNIDOS' THEN 1 ELSE 0 END AS feature_hollywood,
        CASE WHEN PAIS_OBRA = 'BRASIL' THEN 1 ELSE 0 END AS feature_nacional,
        PUBLICO AS label_target_predicao
    FROM workspace.default.silver_ancine
""")
df_features.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_ml_features")

In [0]:
%sql
-- 1. Calculando a data de início de cada filme (primeira exibição)
WITH InicioFilme AS (
    SELECT 
        TITULO_BRASIL,
        DATA_EXIBICAO,
        PUBLICO,
        MIN(DATA_EXIBICAO) OVER(PARTITION BY TITULO_BRASIL) AS DATA_INICIO
    FROM workspace.default.silver_ancine
),

-- 2. Agrupando o público real por Semana de Vida para cada Filme
SemanasReais AS (
    SELECT 
        TITULO_BRASIL,
        -- DATEDIFF divide por 7 para transformar dias em semanas exatas
        FLOOR(DATEDIFF(DATA_EXIBICAO, DATA_INICIO) / 7) + 1 AS SEMANA_DE_VIDA,
        SUM(PUBLICO) AS PUBLICO_TOTAL_SEMANA
    FROM InicioFilme
    GROUP BY TITULO_BRASIL, FLOOR(DATEDIFF(DATA_EXIBICAO, DATA_INICIO) / 7) + 1
),

-- 3. Pegando o público real da PRIMEIRA SEMANA (para servir de base matemática)
BaseSemanaUm AS (
    SELECT 
        TITULO_BRASIL,
        PUBLICO_TOTAL_SEMANA AS PUBLICO_SEMANA_1
    FROM SemanasReais
    WHERE SEMANA_DE_VIDA = 1
)

-- 4. Cruzando as semanas seguintes com a Semana 1 para ver a queda do filme
SELECT 
    s.TITULO_BRASIL,
    s.SEMANA_DE_VIDA,
    s.PUBLICO_TOTAL_SEMANA,
    b.PUBLICO_SEMANA_1,
    ROUND((s.PUBLICO_TOTAL_SEMANA / b.PUBLICO_SEMANA_1) * 100, 2) AS TAXA_RETENCAO_PERCENTUAL
FROM SemanasReais s
JOIN BaseSemanaUm b ON s.TITULO_BRASIL = b.TITULO_BRASIL
-- Filtrando só as 4 primeiras semanas e filmes que estouraram na estreia (pra não sujar a conta com filme nanico)
WHERE s.SEMANA_DE_VIDA BETWEEN 1 AND 4 
  AND b.PUBLICO_SEMANA_1 > 500000 
ORDER BY s.TITULO_BRASIL, s.SEMANA_DE_VIDA;